# Qwen3.5-9B · SQL SFT · Colab

Dùng hai ZIP cùng release, chép vào `MyDrive/DataMateSQLSFT/`. Bộ train được tách theo doanh nghiệp; validation là tập phát triển đã được xem khi audit. Đọc README trong ZIP trước khi chạy.

Mặc định **QLoRA, smoke 1 bước trên mẫu train dài nhất rồi pilot 5 bước**. Chọn GPU và High RAM. A100 là lựa chọn thuận tiện cho context 16K; L4 phải vượt smoke/pilot mới biết có vừa bộ nhớ. T4 16GB bị chặn bởi profile này. Chưa có kết quả training GPU từ bước chuẩn bị dữ liệu.

Để chạy toàn bộ sau pilot, đổi `STAGE="full"`, dùng RUN_TAG mới. Model luôn lấy đúng base revision đã pin. Notebook này không sửa notebook inference đang có.

## v1.3 — kernels và resume
Giữ nguyên data/context/rank/batch/accumulation. FLA 0.6.0 chạy cả DeltaNet và conv Triton; probe forward/backward trước khi load 9B. Giữ Torch/Triton đang có. Mặc định pilot 5 bước; chưa có số đo speedup mới. `RESUME="auto"` dùng checkpoint hoàn tất mới nhất, bỏ qua bản ghi dở. Giữ nguyên RUN_TAG và cấu hình khi nối lại lượt đang train. Full dùng tag mới. Lưu sau bước 1, mỗi 5 bước hoặc ngưỡng 10 phút tại ranh giới bước, giữ 3 bản; evaluation mỗi 50 bước. Phần sau checkpoint gần nhất vẫn có thể phải tính lại. Chỉ chạy một server cho một RUN_TAG.


In [ ]:
from pathlib import Path
METHOD = "qlora"  # "qlora" hoặc "lora"
STAGE = "full"   # pilot: 5 steps; full: 1 epoch
RUN_TAG = "qwen35-9b-sql-v1.3-full-01"
DRIVE_ROOT = Path("/content/drive/MyDrive/DataMateSQLSFT")
SOURCE_ZIP = DRIVE_ROOT / 'datamate-qwen35-9b-sql-source-v1.3.zip'
DATASET_ZIP = DRIVE_ROOT / 'qwen35-9b-sql-sft-v1.3.zip'
EXPECTED_SOURCE_SHA256 = '73565d0e41138d76295afd3fc0d06dca995aa5fdbd10b90aba146ecbb3a5adcf'
BASE_DIR = DRIVE_ROOT / "models/qwen35-9b-c202236"
SEED = 42
assert METHOD in {"qlora","lora"} and STAGE in {"pilot","full"}
assert RUN_TAG and Path(RUN_TAG).name == RUN_TAG

RESUME = "auto"  # None: lượt mới; auto: tiếp từ checkpoint đầy đủ nếu có


In [ ]:
def run_logged(command, *, cwd, log_path):
    """Stream child stdout/stderr into notebook output and an exclusive log."""
    from collections import deque
    from pathlib import Path
    import subprocess
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=60)
    print("Bắt đầu tiến trình; log:", log_path, flush=True)
    with log_path.open("x", encoding="utf-8") as log:
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                if process.poll() is None:
                    process.terminate()
                    try:
                        process.wait(timeout=15)
                    except subprocess.TimeoutExpired:
                        process.kill()
                        process.wait()
                raise
    if returncode:
        raise RuntimeError(f"Training exit={returncode}. Log: {log_path}\n" + "".join(tail))
    return returncode


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import hashlib, json, os, stat, subprocess, sys, zipfile
from pathlib import PurePosixPath
assert SOURCE_ZIP.is_file() and DATASET_ZIP.is_file(), "Chép hai ZIP vào DRIVE_ROOT trước"
with SOURCE_ZIP.open("rb") as handle:
    actual = hashlib.file_digest(handle,"sha256").hexdigest()
assert actual == EXPECTED_SOURCE_SHA256, "Source ZIP sai release"
WORK = Path("/content") / ("datamate-sql-" + actual[:12])
with zipfile.ZipFile(SOURCE_ZIP) as handle:
    assert len(set(handle.namelist())) == len(handle.infolist())
    assert sum(x.file_size for x in handle.infolist()) < 64*1024**2
    for item in handle.infolist():
        path=PurePosixPath(item.filename)
        assert not path.is_absolute() and ".." not in path.parts and "\\" not in item.filename and ":" not in item.filename
        assert not stat.S_ISLNK(item.external_attr >> 16)
    WORK.mkdir(parents=True,exist_ok=True)
    handle.extractall(WORK)
if "datamate" in sys.modules:
    assert Path(sys.modules["datamate"].__file__).resolve().is_relative_to(WORK), "Restart runtime: đang giữ source khác trong RAM"
import importlib.metadata as metadata
torch_version=metadata.version("torch")
constraints=WORK/"colab-torch-constraints.txt"
pins=["torch=="+torch_version]
try: pins.append("triton=="+metadata.version("triton"))
except metadata.PackageNotFoundError: pass
constraints.write_text("\n".join(pins)+"\n",encoding="utf-8")
from uuid import uuid4
run_logged([sys.executable,"-u","-m","pip","install","-c",str(constraints),
            "-r",str(WORK/"configs/sql-qwen35-fast-requirements.txt"),"-e",str(WORK)+"[train]"],
           cwd=WORK,log_path=DRIVE_ROOT/"setup-logs"/("install-"+uuid4().hex+".log"))
assert metadata.version("torch")==torch_version, "Torch thay đổi: dừng và kiểm tra cài đặt"
sys.path.insert(0,str(WORK/"src"))
print("Source:",actual,"Torch được giữ:",torch_version)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bắt đầu tiến trình; log: /content/drive/MyDrive/DataMateSQLSFT/setup-logs/install-cd642acb578341d6a3d4d8b02034c95f.log
Obtaining file:///content/datamate-sql-73565d0e4113
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for datamate-agent (pyproject.toml): started
  Building editable for datamate-agent (pyproject.toml): finished with status 'done'
  Created wheel for datamate-agent: filename=datamate_

In [ ]:
from uuid import uuid4
import torch
from datamate.training.colab import gpu_preflight
from datamate.training.sql_dataset import unpack_sql_archive
from datamate.models.artifacts import stage_model, verify_artifact
from datamate.models.registry import get_model
gpu = gpu_preflight(torch,min_vram_gib=22 if METHOD=="qlora" else 38)
print(json.dumps(gpu,ensure_ascii=False,indent=2))
DATASET = WORK/"sql_dataset"
manifest=unpack_sql_archive(DATASET_ZIP,DATASET,WORK/"configs/qwen35-9b-sql-release.json")
print("Verified examples:",manifest["train_examples"],"train,",manifest["validation_examples"],"validation")
# Online staging is separate from training/inference. No full model is loaded here.
os.environ.pop("HF_HUB_OFFLINE",None)
os.environ.pop("TRANSFORMERS_OFFLINE",None)
if not (BASE_DIR/"artifact_manifest.json").is_file():
    stage_model("qwen35-9b",BASE_DIR)
# Copy from persistent Drive to local SSD, then verify the complete sealed artifact.
# No marker is placed inside the sealed model directory.
import shutil, time
drive_base=BASE_DIR
base_sha=hashlib.sha256((drive_base/"artifact_manifest.json").read_bytes()).hexdigest()
BASE_DIR=Path("/content/datamate-models")/("qwen35-9b-"+base_sha[:16])
local_valid=False
if (BASE_DIR/"artifact_manifest.json").is_file():
    try:
        verify_artifact(BASE_DIR,"qwen35-9b")
        local_valid=True
    except Exception as exc:
        print("Local model cache chưa hoàn tất, chép lại từ Drive:",exc)
copy_started=time.monotonic()
if not local_valid:
    base_manifest=json.loads((drive_base/"artifact_manifest.json").read_text(encoding="utf-8"))
    needed=sum(item["bytes"] for item in base_manifest["files"].values())
    assert shutil.disk_usage("/content").free>needed, "Thiếu local disk để stage model"
    BASE_DIR.mkdir(parents=True,exist_ok=True)
    for name in [*base_manifest["files"],"artifact_manifest.json"]:
        original=(drive_base/name).resolve()
        target=(BASE_DIR/name).resolve()
        assert original.is_relative_to(drive_base.resolve()) and target.is_relative_to(BASE_DIR.resolve())
        target.parent.mkdir(parents=True,exist_ok=True)
        print("Copy model:",name,flush=True)
        shutil.copy2(original,target)
    verify_artifact(BASE_DIR,"qwen35-9b")
print("Model local đã xác minh:",BASE_DIR,"; copy/verify seconds:",round(time.monotonic()-copy_started,2))
print("Pinned base:",get_model("qwen35-9b").to_dict())
OUT = DRIVE_ROOT/"runs"/RUN_TAG
OUT.mkdir(parents=True,exist_ok=True)
invocation_id=uuid4().hex
(OUT/("environment-"+invocation_id+".json")).write_text(json.dumps({"gpu":gpu,"source_sha256":EXPECTED_SOURCE_SHA256,
    "dataset_manifest_sha256":hashlib.sha256((DATASET/"manifest.json").read_bytes()).hexdigest(),
    "base":get_model("qwen35-9b").to_dict(),"method":METHOD,"stage":STAGE},ensure_ascii=False,indent=2),encoding="utf-8")
freeze=subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)
(OUT/("pip-freeze-"+invocation_id+".txt")).write_text(freeze,encoding="utf-8")

from uuid import uuid4
kernel_report=OUT/("kernel-check-"+uuid4().hex+".json")
run_logged([sys.executable,"-u","-m","datamate.training.kernels","--output",str(kernel_report)],
           cwd=WORK,log_path=OUT/"logs"/("kernel-check-"+uuid4().hex+".log"))
print("Kernel forward/backward report:",kernel_report)


{
  "gpu": "NVIDIA A100-SXM4-40GB",
  "total_vram_gib": 39.49,
  "bf16_supported": true,
  "torch": "2.14.1+cu130",
  "cuda": "13.0",
  "minimum_vram_policy_gib": 22,
  "memory_fit_verified": false,
  "next_step": "one optimizer step on the longest actual sequence"
}
Verified examples: 2347 train, 618 validation
Model local đã xác minh: /content/datamate-models/qwen35-9b-f8b32a3f0b36609c ; copy/verify seconds: 0.0
Pinned base: {'key': 'qwen35-9b', 'repo_id': 'Qwen/Qwen3.5-9B', 'revision': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a', 'parameters': 9653104368, 'revision_date': '2026-03-02T00:51:43Z', 'license': 'apache-2.0', 'assistant_for': None}
Bắt đầu tiến trình; log: /content/drive/MyDrive/DataMateSQLSFT/runs/qwen35-9b-sql-v1.3-full-01/logs/kernel-check-b8ae575be2ff4664aff2e535db524b35.log
{
  "device": "NVIDIA A100-SXM4-40GB",
  "torch": "2.14.1+cu130",
  "cuda": "13.0",
  "versions": {
    "transformers": "5.22.1",
    "peft": "0.21.2",
    "accelerate": "1.15.0",
    "triton": "3.

In [ ]:
profile=json.loads((WORK/"configs"/("sql-qwen35-9b-"+METHOD+".json")).read_text(encoding="utf-8"))
profile.update(base_dir=str(BASE_DIR),dataset_dir=str(DATASET),seed=SEED,
               output_dir=str(OUT/"adapter"),max_length=manifest["max_length"],
               max_steps=5 if STAGE=="pilot" else -1)
smoke={**profile,"output_dir":str(OUT/"smoke_adapter"),"max_steps":1,
       "gradient_accumulation_steps":1,"smoke_longest_only":True,"save_steps":1}
from uuid import uuid4
completed = None
for candidate in [OUT/"smoke_adapter", *sorted(OUT.glob("smoke_adapter_retry_*"))]:
    report_path = candidate/"training_report.json"
    expected = {**smoke, "output_dir":str(candidate)}
    if report_path.is_file() and json.loads(report_path.read_text(encoding="utf-8"))["training_config"] == expected:
        smoke, completed = expected, candidate
        break
if completed is None:
    candidate = OUT/"smoke_adapter"
    if candidate.exists() and any(candidate.iterdir()):
        candidate = OUT/("smoke_adapter_retry_"+uuid4().hex[:12])
    smoke["output_dir"] = str(candidate)
    smoke_path=OUT/(candidate.name+"_config.json")
    smoke_path.write_text(json.dumps(smoke,indent=2),encoding="utf-8")
    run_logged([sys.executable,"-u","-m","datamate","train","--config",str(smoke_path)],
               cwd=WORK,log_path=OUT/"logs"/("smoke-"+uuid4().hex+".log"))
    report_path=candidate/"training_report.json"
    assert report_path.is_file(), "Tiến trình kết thúc nhưng thiếu training_report.json; xem log"
    assert json.loads(report_path.read_text(encoding="utf-8"))["training_config"] == smoke
print("Smoke report:",smoke["output_dir"],"— child process đã kết thúc.")


Bắt đầu tiến trình; log: /content/drive/MyDrive/DataMateSQLSFT/runs/qwen35-9b-sql-v1.3-full-01/logs/smoke-5dfe68ad012a4903a761faeaa3078c0a.log
Training kernel dispatch: {"delta_rule": {"external": true, "implementation": "fla.ops.gated_delta_rule.chunk.chunk_gated_delta_rule"}, "causal_conv1d": {"external": true, "implementation": "fla.modules.causal_conv1d.ops.causal_conv1d", "backend": "triton", "layout_adapter": "B,D,T <-> B,T,D"}}

Loading weights: 100%|██████████| 760/760 [00:05<00:00, 143.76it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.

100%|██████████| 1/1 [00:07<00:00,  7.13s/it]
                                             
{'loss': '0.3795', 'grad_norm': '1.078', 'learning_rate': '0', 'epoch': '1'}

100%|██████████| 1/1 [00:07<00:00,

In [ ]:
config_path=OUT/"training_config.json"
from datamate.training.checkpoints import completed_report, ensure_training_config
ensure_training_config(config_path,profile)
from uuid import uuid4
report=completed_report(OUT/"adapter",profile)
if report is None:
    command=[sys.executable,"-u","-m","datamate","train","--config",str(config_path)]
    if RESUME is not None: command += ["--resume",str(RESUME)]
    run_logged(command,cwd=WORK,log_path=OUT/"logs"/("train-"+uuid4().hex+".log"))
    report=completed_report(OUT/"adapter",profile)
    assert report is not None, "Thiếu báo cáo hoàn tất; xem log/checkpoint"
else:
    print("Lượt train này đã hoàn tất; hiển thị kết quả đã xác minh, không chạy lại.")
print(json.dumps({k:report.get(k) for k in ["global_step","train_sequences","validation_sequences",
    "trainable_parameters","peak_cuda_bytes","metrics","kernel_dispatch"]},ensure_ascii=False,indent=2))
print("Adapter và checkpoint trên Drive:",OUT/"adapter")
print("Sau pilot: đổi STAGE='full' và RUN_TAG mới. Sau ngắt kết nối: giữ cùng RUN_TAG, STAGE và cấu hình để resume.")


Streaming output truncated to the last 5000 lines.

 60%|██████    | 179/298 [2:06:42<1:00:26, 30.47s/it]
                                                     
{'loss': '0.03189', 'grad_norm': '0.5155', 'learning_rate': '2.076e-05', 'epoch': '0.6012'}

 60%|██████    | 180/298 [2:07:10<58:48, 29.90s/it]  
                                                   
{'loss': '0.02976', 'grad_norm': '0.6546', 'learning_rate': '2.059e-05', 'epoch': '0.6045'}

 60%|██████    | 180/298 [2:07:10<58:48, 29.90s/it]Checkpoint hoàn tất: /content/drive/MyDrive/DataMateSQLSFT/runs/qwen35-9b-sql-v1.3-full-01/adapter/checkpoint-180

 61%|██████    | 181/298 [2:07:44<1:00:23, 30.97s/it]
                                                     
{'loss': '0.00174', 'grad_norm': '0.1976', 'learning_rate': '2.042e-05', 'epoch': '0.6079'}

 61%|██████    | 182/298 [2:08:13<58:53, 30.46s/it]  
                                                   
{'loss': '0.006102', 'grad_norm': '0.5489', 'learning_rate': '2.024e-05', '

In [ ]:
import json

folder = OUT / "adapter"
report_path = folder / "training_report.json"

print("Thư mục:", folder)
print("File trọng số:", [p.name for p in folder.glob("adapter_model.*")])
print("Có báo cáo hoàn tất:", report_path.is_file())

if report_path.is_file():
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Số bước cấu hình:", report["training_config"].get("max_steps"))
    print("Khớp cấu hình hiện tại:", report["training_config"] == profile)
    print(json.dumps(report["metrics"], indent=2))
else:
    print("Chưa xác nhận hoàn tất; cần xem log trong OUT/logs.")

Thư mục: /content/drive/MyDrive/DataMateSQLSFT/runs/qwen35-9b-sql-v1.3-full-01/adapter
File trọng số: ['adapter_model.safetensors']
Có báo cáo hoàn tất: True
Số bước cấu hình: -1
Khớp cấu hình hiện tại: True
{
  "train_runtime": 13353.0217,
  "train_samples_per_second": 0.178,
  "train_steps_per_second": 0.022,
  "total_flos": 1.2341440686832584e+18,
  "train_loss": 0.04495120244802893,
  "epoch": 1.0
}


Training loss và validation loss không thay cho chất lượng agent. Trước khi chốt adapter, chạy baseline và bản fine-tune trên cùng validation IDs, cùng code/prompt/mode/ngân sách; so PA, sai, abstain, số calls và thời gian. Dataset dùng `mode=single` với auto-execute; không lấy PA self-review cũ làm baseline tương đương. Giữ adapter cùng base revision; merge/seal bằng CLI `datamate merge` trên GPU đủ bộ nhớ rồi mới đưa vào inference offline.